# F5 — the effect of shrinking the NMS radius, with border padding held fixed

**What this notebook is.** `tm_threshold_axis_sweep_v2.ipynb` changed **two** things at once
relative to v1 — border padding *and* a smaller NMS radius — and said so in its own cost table:

> Note "the cost of *both* fixes": v1 differs from this run by padding *and* radius, so this
> delta cannot be attributed to the radius alone.

This notebook attributes it. Padding is held **ON in every arm**; the NMS radius is the only
thing that moves.

**The two variants, as asked for.**

| | | radius | px over the 14 ROIs |
|---|---|---|---|
| **Variant 1** | `r7.5` — border padding only, **default** suppression | 7.5 µm = `evaluate.MIDOG_RADIUS_UM` | 29.61 – 33.14 |
| **Variant 2a** | padding + a **mild** shrink | 5.9 µm | 23.29 – 26.07 |
| **Variant 2b** | padding + a **larger** shrink (v2's value) | 5.0 µm | 19.74 – 22.09 |

Variant 1 is not a strawman and not a re-run: it is **the arm that has never existed**. v1 had
this radius but no padding; v2 had padding but not this radius.

**Why one run and not two notebooks.** The expensive step — the fused response and the deep peak
pool — is entirely radius-independent, so `f5_nms_radius_ablation.py` computes it *once* per
(ROI, seed) and applies all three radii to the identical `(centers, scores)` arrays. The arms
differ by exactly one argument, and every comparison below is a within-(ROI, seed, z, axis)
**paired** difference. Two separate runs would triple the compute and destroy the pairing.

**Sample.** 14 ROIs (`images/extra_valid`, 2 per tumour type across all 7 domains) × 5 seeds =
70 paired cells, clustered in 14 ROIs. This fixes both limitations `tm_threshold_axis_sweep_v2`
names about itself: one ROI per domain, and `SEED_INDEX = 0` only.

**Pre-registered** in [`Research Logs/2026-09-04-f5-preregistration.md`](Research%20Logs/2026-09-04-f5-preregistration.md),
revision 3, converged after two rounds of adversarial review. Every design choice below has a
numbered justification there; §11 and §12 record what review changed and what the unrevised
version would have produced. **Read the verification section below before any table.**

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

RES     = pd.read_csv("results/f5_nms_radius_ablation.csv")
VERIF   = pd.read_csv("results/f5_nms_radius_ablation_verification.csv")
LEDGER  = pd.read_csv("results/f5_nms_radius_ablation_ledger.csv")
SPACING = pd.read_csv("results/f5_nms_radius_ablation_spacing.csv")

CONTROL, DOSES = "r7.5", ("r5.9", "r5.0")
TAGS = (CONTROL,) + DOSES
PRIMARY_AXIS, HEADLINE_Z, HEADLINE_K = "chromatin_od", 1.0, 250

# D4 data trap: the CSV emits one row per budget, so every arm-level column repeats 18 times.
# The dedup key must include the axis -- which is why the arms are named `{axis}@{radius}`.
DEDUP = ["file_name", "seed_index", "arm", "z"]

def cells(df=RES):
    """One row per (ROI, seed, arm, z) -- for arm-level columns only."""
    return df.drop_duplicates(DEDUP).copy()

print(f"{RES.shape[0]} rows, {len(cells())} (ROI, seed, arm, z) cells, "
      f"{RES['file_name'].nunique()} ROIs x {RES['seed_index'].nunique()} seeds")
print("arms:", sorted(RES["arm"].unique()))
print("z   :", sorted(RES["z"].unique()))
print("K   :", sorted(RES["budget"].unique()))

## 1. Verification first

`evaluate.py`'s rule is to read `coverage_frac` before any full-list number; this repo's rule is
to read `*_verification.csv` before any table at all. The two checks that could actually have
caught a silent error in *this* run are:

* **`check_nms_radius` on the control** — a *positive* control that variant 1 really is the repo
  default. The shrunk arms pass `nms_radius=None`, because the check is deliberately inapplicable
  to them; that is the invariant this experiment overturns on purpose, not one it fails.
* **the v2 reproduction gate** — `r5.0` at seed 0 must reproduce
  `results/tm_ccoeff_threshold_axis_sweep_v2.csv` exactly on the 7 overlapping ROIs. `draw_seeds(0)`
  is byte-identical to v2's inline draw and `DEEP_FLOOR_Z` is pinned at v2's −1.5, so this is the
  only check that can catch a divergence in the *whole pipeline* rather than in one function.

`check_min_separation` is the direct proof that the suppression each arm claims is the suppression
that ran, and the shortcut check confirms that filtering one deep pool at each `z` equals
re-extracting at that `z` — verified at **z = 3.0 as well as z = 1.0**, because z = 1.0 drops only
~15 % of the pool and the one historical failure of that shortcut was found at z = 2.

In [ ]:
VERIF["passed"] = VERIF["passed"].astype(bool)
display(VERIF.groupby("check")["passed"]
             .agg(n="size", passed="sum")
             .assign(all_passed=lambda d: d["n"] == d["passed"])
             .reset_index())

failed = VERIF[~VERIF["passed"]]
print(f"{len(failed)} failed check(s)")
if len(failed):
    display(failed.dropna(axis=1, how="all"))

print("\nv2 reproduction gate (r5.0 @ seed 0 vs results/tm_ccoeff_threshold_axis_sweep_v2.csv):")
display(VERIF[VERIF["check"] == "v2_reproduction"].dropna(axis=1, how="all"))

print("\ncheck_nms_radius -- ran on the CONTROL arm only, as a positive control:")
display(VERIF[VERIF["check"] == "nms_radius"].dropna(axis=1, how="all").head())

## 2. The geometry that decided the two shrink values

Measured over all 14 ROIs, in **microns** — the only unit in which a µm-scaled radius can be
compared across scanners whose mpp runs 0.2263–0.2533.

This table is also why the ladder is registered as a **dose–response on one mechanism** rather
than a mechanism decomposition. Across ~1,325 mitotic annotations, exactly **one** mitotic pair is
close enough for the *default* radius to merge, 12 of 14 ROIs have zero pairs inside it, and
**zero** pairs lie between 5.0 and 5.9 µm — so the two shrink levels are identical with respect to
un-merging neighbouring ground truth. Whatever the shrink does, it is not that.

In [ ]:
cols = ["file_name", "tumor_type", "mpp", "n_mitotic", "min_mit_mit_um", "min_any_any_um",
        "r7.5_px", "r5.9_px", "r5.0_px",
        "r7.5_pairs_mit", "r7.5_pairs_any", "r5.9_pairs_any", "r5.0_pairs_any"]
display(SPACING[cols].round(4))
print(f"closest any-any pair : {SPACING['min_any_any_um'].min():.4f} um "
      f"({SPACING.loc[SPACING['min_any_any_um'].idxmin(), 'file_name']})")
print(f"closest mit-mit pair : {SPACING['min_mit_mit_um'].min():.4f} um "
      f"({SPACING.loc[SPACING['min_mit_mit_um'].idxmin(), 'file_name']})")
for t in TAGS:
    print(f"  {t}: GT pairs inside -> {SPACING[f'{t}_pairs_mit'].sum()} mitotic / "
          f"{SPACING[f'{t}_pairs_any'].sum()} any")

## 3. The coverage guard — read this before any full-list number

`evaluate.py`: *"A detection list long enough to tile the ROI answers 'is this annotation within
the match radius of some detection?' by geometry rather than by evidence."* A **smaller** NMS
radius keeps more peaks, which raises coverage, which mechanically raises full-list recall. So a
full-recall gain that arrives with a coverage rise is not evidence about suppression.

The consequence here is sharp: wherever coverage is near 1, `full_list_recall`, `read_99`,
`read_100` and `FP_to_1.0` are all reporting the same geometry under different names. Only where
coverage is well below 1 can any of them carry evidence.

In [ ]:
cv = cells()
cv = cv[cv["arm"].str.startswith(PRIMARY_AXIS)]
print("coverage_frac -- fraction of ARBITRARY ROI locations already within a match radius of")
print("some detection (mean over 14 ROIs x 5 seeds):")
display(cv.pivot_table(index="nms_radius_tag", columns="z", values="coverage_frac",
                       aggfunc="mean").reindex(TAGS).round(3))
rng = cv.groupby("z")["coverage_frac"].agg(["min", "max"]).round(3)
print("across all arms and cells, by z:")
display(rng)
usable = rng[rng["max"] < 0.8]
print("z levels where coverage stays below 0.8 (full-recall numbers can carry evidence): "
      f"{list(usable.index) if len(usable) else 'none'}")

## 4. The headline reporting frame (D4) — worst ROI, worst click, per domain

`DECISIONS.md` D4: *"the smallest K such that recall@K ≥ target on the worst ROI and the worst
click, reported per tumour domain."* Two ROIs per domain × 5 seeds = **worst-of-10** per domain.

This is a **descriptive** frame and carries no p-value. D4's rule is about how to *present*
results so a domain-specific collapse cannot hide behind an average; it is not an inference rule,
and the statistic that arbitrates is a different one (§5). Treating the two as the same quantity
is a mistake an earlier revision of the pre-registration made and this one corrects.

In [ ]:
def recall_at(k=HEADLINE_K, z=HEADLINE_Z, axis=PRIMARY_AXIS):
    d = RES[(RES["budget"] == k) & (RES["z"] == z) & (RES["arm"].str.startswith(axis))]
    return d.drop_duplicates(["file_name", "seed_index", "arm"])

r = recall_at()
worst = (r.groupby(["tumor_type", "nms_radius_tag"])["recall_at_budget"].min()
          .unstack("nms_radius_tag").reindex(columns=list(TAGS)))
print(f"WORST-OF-10 recall@{HEADLINE_K}, z={HEADLINE_Z}, {PRIMARY_AXIS} axis, per domain:")
display(worst.round(4))
print("difference of worsts -- DESCRIPTIVE ONLY. It is unpaired; see section 5:")
display(worst[list(DOSES)].sub(worst[CONTROL], axis=0).round(4))

print(f"\nceiling check -- recall@{HEADLINE_K} can be pinned near 1.0 on the sparse ROIs:")
display(r.groupby("file_name")
         .agg(tumor_type=("tumor_type", "first"), n_gt_mitotic=("n_gt_mitotic", "first"),
              min_recall=("recall_at_budget", "min"), max_recall=("recall_at_budget", "max"))
         .sort_values("n_gt_mitotic").round(4))

## 5. The arbiter — the mean-of-5 **paired** Δ per ROI

> **The arbiter is the mean-of-5 paired Δ(recall@250) per ROI**, at z = 1.0 on the
> `chromatin_od` axis: form the *paired* difference within each cell (same ROI, same seed, same
> peak pool), average the 5 within an ROI, and treat the **14 values as clustered units**.

**Why not the worst-of-5 that §4 shows.** `worst-of-5(shrink) − worst-of-5(control)` takes each
arm's minimum *independently*, so the argmin need not be the same seed in both arms — which
discards exactly the pairing the design exists to create. On `results/tm_nms_radius_sweep.csv`
(`read_50`, 29.6 → 20.0 px) the two disagree by 3 on 300.tiff and by 36 on 301.tiff, with
different argmin seeds. It is also biased in a direction: the minimum of 5 draws falls as variance
rises, and the shrunk arm has the longer, more seed-dependent list.

**Why clusters and not cells.** Five seeds share an ROI — one tissue, one ground truth, one
response surface — and two ROIs share a domain. The treatment acts on the response surface, which
is an ROI property, so the 70 cells are not 70 independent units and a cell-level test is
anti-conservative by an unknown factor. The cell-level Wilcoxon is reported anyway, explicitly
labelled as that bound, so the size of the gap is visible.

Multiplicity: **Holm across the 2 arbiter tests** (`r5.9`, `r5.0` on `chromatin_od`). `tm_score`
is a secondary axis, uncorrected. Both secondary worst-case statistics are pre-specified and
reported unconditionally, whatever they show.

In [ ]:
def paired_delta(k=HEADLINE_K, z=HEADLINE_Z, axis=PRIMARY_AXIS, value="recall_at_budget"):
    """One row per (ROI, seed), control and each dose side by side -- the paired unit."""
    d = RES[(RES["budget"] == k) & (RES["z"] == z) & (RES["arm"].str.startswith(axis))]
    d = d.drop_duplicates(["file_name", "seed_index", "arm"])
    w = d.pivot_table(index=["tumor_type", "file_name", "seed_index"],
                      columns="nms_radius_tag", values=value)
    for t in DOSES:
        w[f"d_{t}"] = w[t] - w[CONTROL]
    return w.reset_index()

def signflip_p(cluster_means):
    """Exact two-sided cluster sign-flip permutation: 2**14 = 16384 assignments, enumerated."""
    x = np.asarray(cluster_means, dtype=float)
    signs = np.array(list(itertools.product([1, -1], repeat=len(x))))
    perm = (signs * x).mean(axis=1)
    return float((np.abs(perm) >= abs(x.mean()) - 1e-15).mean()), int(len(signs))

def cluster_boot_ci(cluster_means, n_boot=10000, seed=20260904):
    x = np.asarray(cluster_means, dtype=float)
    idx = np.random.default_rng(seed).integers(0, len(x), size=(n_boot, len(x)))
    m = x[idx].mean(axis=1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def holm(pvals):
    p = np.asarray(pvals, float)
    adj, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj

W = paired_delta()
rows, raw_p = [], []
for t in DOSES:
    cell = W[f"d_{t}"].to_numpy()
    roi_mean = W.groupby("file_name")[f"d_{t}"].mean().to_numpy()
    p, n_perm = signflip_p(roi_mean)
    lo, hi = cluster_boot_ci(roi_mean)
    wil = stats.wilcoxon(cell, zero_method="zsplit").pvalue if np.any(cell != 0) else np.nan
    rows.append({"dose": t, "n_clusters": len(roi_mean), "n_cells": len(cell),
                 "mean_paired_delta": roi_mean.mean(), "boot_ci_lo": lo, "boot_ci_hi": hi,
                 "signflip_p": p, "n_perm": n_perm,
                 "cell_wilcoxon_p_ANTICONSERVATIVE": wil})
    raw_p.append(p)
ARB = pd.DataFrame(rows)
ARB["holm_p"] = holm(raw_p)
print(f"ARBITER -- mean-of-5 paired delta(recall@{HEADLINE_K}) per ROI, z={HEADLINE_Z}, "
      f"{PRIMARY_AXIS}, n=14 clusters, Holm across 2:")
display(ARB.round(6))

print("\nSECONDARY, pre-specified, paired, reported unconditionally, no corrected p:")
sec = []
for t in DOSES:
    g = W.groupby("file_name")
    worst_paired = g[f"d_{t}"].min()
    at_ctrl_argmin = g.apply(lambda x: x.loc[x[CONTROL].idxmin(), f"d_{t}"])
    sec.append({"dose": t,
                "median_worst_paired_delta_in_roi": worst_paired.median(),
                "n_rois_worst_paired_negative": int((worst_paired < 0).sum()),
                "median_delta_at_control_argmin_seed": at_ctrl_argmin.median(),
                "n_rois_negative_at_control_argmin": int((at_ctrl_argmin < 0).sum())})
display(pd.DataFrame(sec).round(6))

print("\nthe 14 clustered units (mean paired delta per ROI):")
display(W.groupby(["tumor_type", "file_name"])[[f"d_{t}" for t in DOSES]].mean().round(4))

print("\nsame arbiter on the SECONDARY axis (tm_score, uncorrected):")
W2 = paired_delta(axis="tm_score")
display(pd.DataFrame([{
    "dose": t, "mean_paired_delta": W2.groupby("file_name")[f"d_{t}"].mean().mean(),
    "signflip_p": signflip_p(W2.groupby("file_name")[f"d_{t}"].mean().to_numpy())[0]}
    for t in DOSES]).round(6))

## 6. recall@K, and the crossover K\*

The prior 2-ROI sweep says the expected result is a **trade, not a win**: on 301.tiff `read_50`
degrades monotonically with the shrink on all five seeds, while every seed that had misses reaches
zero at every shrink level. Better tail, worse head — which means there is a **crossover K**, and
a design sampling only K = 250 could sit on either side of it and see nothing.

K\* is therefore an output in its own right, not a by-product. If K\* > 5000, the shrink is
irrelevant to the product: no pathologist reads 5000 candidates.

**Guard:** `budget_delivered == budget`. The control has the *shortest* list and so exhausts
first; a K where it has run out makes it look flat for a reason unrelated to suppression.

In [ ]:
d = RES[(RES["z"] == HEADLINE_Z) & (RES["arm"].str.startswith(PRIMARY_AXIS))]
exh = d[d["budget_delivered"] < d["budget"]]
print(f"cells where the list was exhausted before K ({len(exh)} of {len(d)}):")
if len(exh):
    display(exh.groupby(["nms_radius_tag", "budget"]).size().unstack(fill_value=0))
else:
    print("  none -- every reported K is fully delivered at this z")

uniq = d.drop_duplicates(["file_name", "seed_index", "arm", "budget"])
curve = (uniq.groupby(["tumor_type", "nms_radius_tag", "budget"])["recall_at_budget"]
             .min().reset_index())            # worst-of-10 per domain, D4

Ks = sorted(d["budget"].unique())
cell_curve = uniq.pivot_table(index=["tumor_type", "file_name", "seed_index", "budget"],
                              columns="nms_radius_tag", values="recall_at_budget")

def crossover(ctrl, dose):
    for k in Ks:
        a, b = ctrl.get(k, np.nan), dose.get(k, np.nan)
        if np.isfinite(a) and np.isfinite(b) and b > a:
            return k
    return np.nan

kstar = []
for (dom, fn, si), g in cell_curve.groupby(level=[0, 1, 2]):
    gg = g.droplevel([0, 1, 2])
    for t in DOSES:
        kstar.append({"tumor_type": dom, "file_name": fn, "seed_index": si, "dose": t,
                      "K_star": crossover(gg[CONTROL], gg[t])})
KS = pd.DataFrame(kstar)
print(f"\nK* -- smallest K where the dose beats the control, per cell, z={HEADLINE_Z}:")
display(KS.groupby("dose")["K_star"].agg(
    n="size", n_with_crossover=lambda s: int(s.notna().sum()),
    median="median", q1=lambda s: s.quantile(.25), q3=lambda s: s.quantile(.75)).round(1))
print("by domain (median K*; NaN = the dose never overtakes anywhere in the grid):")
display(KS.pivot_table(index="tumor_type", columns="dose", values="K_star",
                       aggfunc="median", dropna=False))

In [ ]:
domains = sorted(curve["tumor_type"].unique())
nrows = -(-len(domains) // 2)
fig, axes = plt.subplots(nrows, 2, figsize=(12, 3.4 * nrows))
axes = np.atleast_1d(axes).ravel()
colors = {CONTROL: "#333333", "r5.9": "#4c72b0", "r5.0": "#c44e52"}
styles = {CONTROL: "-", "r5.9": "--", "r5.0": "-."}
labels = {CONTROL: "variant 1: r7.5 (default)", "r5.9": "variant 2a: r5.9",
          "r5.0": "variant 2b: r5.0"}
for ax, dom in zip(axes, domains):
    for t in TAGS:
        s = curve[(curve["tumor_type"] == dom) & (curve["nms_radius_tag"] == t)]
        ax.plot(s["budget"], s["recall_at_budget"], color=colors[t], linestyle=styles[t],
                marker="o", markersize=2.5, linewidth=1.5, label=labels[t])
    ax.axvline(HEADLINE_K, color="#bbbbbb", linestyle=":", linewidth=1.0)
    ax.set_xscale("log")
    ax.set_xlabel("K (candidates read)")
    ax.set_ylabel("worst-of-10 recall@K")
    ax.set_title(dom, fontsize=9.5)
    ax.set_ylim(0, 1.02)
for ax in axes[len(domains):]:
    ax.set_visible(False)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.03), fontsize=9)
fig.suptitle(f"recall@K, worst ROI and worst click per domain, z={HEADLINE_Z}, {PRIMARY_AXIS}",
             y=1.06, fontsize=12)
fig.tight_layout()
plt.show()

## 7. Full-list recall — only ever next to its coverage

Reported because it is what v2 used as its headline, and reported *only* beside `coverage_frac`
because a shrink raises pool size → raises coverage → mechanically raises full-list recall. Per
§3, the only z levels where this number can carry evidence are those where coverage stays well
below 1.

In [ ]:
fl = cells()
fl = fl[fl["arm"].str.startswith(PRIMARY_AXIS)]
g = (fl.groupby(["z", "nms_radius_tag"])
       .agg(full_recall=("full_list_recall", "mean"),
            worst_full_recall=("full_list_recall", "min"),
            coverage=("coverage_frac", "mean")).reset_index())
display(g.pivot(index="z", columns="nms_radius_tag",
                values=["full_recall", "worst_full_recall", "coverage"]).round(4))
print("Read each full_recall cell against the coverage cell directly below it.")

## 8. False positives — three questions the phrase "fewer FPs" hides

1. **At fixed K.** `FP@K = budget_delivered − tp_at_budget`. When the list is not exhausted —
   every headline cell — `precision@K = recall@K · n_mit / K`, an affine restatement of recall@K.
   This is **the same question as §5**, and is labelled as such rather than presented as
   independent corroboration.
2. **Over the full list.** A shrink **necessarily** increases this: it suppresses less, so more
   candidates survive. The honest number is the `n_detections` ratio to control. If "fewer FPs
   overall" means this, the answer is no, and it was known before the run.
3. **FPs paid to reach a recall target** — the dual of recall@K, and the literal form of "most TPs
   per candidate read". `FP_to_r = read_r − ⌈r·n_mit⌉`. **`FP_to_0.99` and `FP_to_1.0` carry the
   same coverage guard as §7** — `read_100` is finite only because the list tiles the ROI, so an
   unguarded improvement there is §7's geometry arriving under a different name. `FP_to_0.8` and
   `FP_to_0.9` are the ones that can carry a claim.

In [ ]:
c = cells()
c = c[(c["arm"].str.startswith(PRIMARY_AXIS)) & (c["z"] == HEADLINE_Z)]

print("(2) FULL LIST -- candidate count per ROI, and the ratio to control:")
nd = (c.groupby(["tumor_type", "nms_radius_tag"])["n_detections"].mean()
       .unstack().reindex(columns=list(TAGS)))
for t in DOSES:
    nd[f"ratio_{t}"] = (nd[t] / nd[CONTROL]).round(3)
display(nd.round(1))

print(f"\n(1) AT FIXED K={HEADLINE_K} -- FP@K = delivered - TP@K "
      "(a monotone restatement of recall@K, NOT independent evidence):")
k = RES[(RES["budget"] == HEADLINE_K) & (RES["z"] == HEADLINE_Z) &
        (RES["arm"].str.startswith(PRIMARY_AXIS))].drop_duplicates(
            ["file_name", "seed_index", "arm"])
k = k.assign(fp_at_k=k["budget_delivered"] - k["tp_at_budget"])
display(k.groupby(["tumor_type", "nms_radius_tag"])
         [["tp_at_budget", "fp_at_k", "lookalike_at_budget"]].mean().unstack().round(1))

print("\n(3) FPs PAID TO REACH A RECALL TARGET (mean over cells; NaN = target unreachable):")
out = []
for r_ in (0.8, 0.9, 0.95, 0.99, 1.0):
    col = f"read_{int(round(r_ * 100))}"
    if col not in c.columns:
        continue
    tmp = c.assign(fp_to_r=c[col] - np.ceil(r_ * c["n_gt_mitotic"]))
    row = tmp.groupby("nms_radius_tag")["fp_to_r"].mean().reindex(list(TAGS))
    row["n_unreached"] = int(tmp[col].isna().sum())
    row["coverage_guarded"] = "YES -- see section 7" if r_ >= 0.99 else "no"
    out.append(row.rename(f"FP_to_{r_}"))
display(pd.DataFrame(out))

## 9. The ledger — does a shrink *gain* mitoses, or just shuffle them?

Greedy NMS is **not monotone in the radius**. A point kept at radius R can be lost at r < R,
because a smaller radius lets an intermediate competitor survive and suppress it — verified by
running `nms_by_distance` itself on A(10), B(9), C(8) with |AB| = 25, |BC| = 15, |AC| = 30:
`keep(29.6) = {A, C}` but `keep(20) = {A, B}`. So "shrinking can only help recall" is false as
stated, and a net gain of 5 made of 8 gains and 3 losses is a materially different finding from
one made of 5 clean gains.

Also here: **`n_dup_fp` with its denominator**. A second detection inside an already-claimed
object's match circle is bucketed as an unannotated false positive — the exact defect
`invariants.check_nms_radius` exists to prevent. The raw count rises mechanically with list
length, so it is reported as a *fraction* and does not arbitrate.

And **`topk_churn`**: without it, a null at K = 250 is uninterpretable — an inert intervention and
an active-but-neutral one look identical.

In [ ]:
L = LEDGER[LEDGER["axis"] == PRIMARY_AXIS]
wl = L[L["gained_vs_control"] >= 0]           # -1 marks the control's own rows
print("WIN / LOSS against the control (z=1.0, chromatin_od, summed over all cells):")
display(wl.groupby("nms_radius_um")[["gained_vs_control", "lost_vs_control",
                                     "gt_claim_change"]].sum().astype(int))
print("\nper-domain net (gained - lost):")
net = wl.assign(net=wl["gained_vs_control"] - wl["lost_vs_control"])
display(net.pivot_table(index="tumor_type", columns="nms_radius_um", values="net",
                        aggfunc="sum"))

print("\nDUPLICATE FPs inside a GT match circle, with the denominator that makes them")
print("comparable across arms of different list length:")
display(L.groupby("nms_radius_um")[["n_detections", "n_within_match_radius", "n_dup_fp",
                                    "dup_fp_frac"]].mean().round(4))

print(f"\nTOP-{HEADLINE_K} CHURN vs the control (1.0 = disjoint lists, 0.0 = identical).")
print("A null at K with near-zero churn would mean an inert intervention, not a neutral one:")
display(L[L["topk_churn"].notna()].groupby("nms_radius_um")["topk_churn"]
         .agg(["mean", "min", "max"]).round(4))

## 10. Is the dose grid actually centred on the mechanism?

The whole ladder is positioned against a **23–29 px same-object suppression band** — the gap
between a mitotic figure's own on-centre peak and the off-centre peak that ate it. That band was
measured on **one ROI and one seed** (v2 cells 9–11) and until now carried the entire
justification for where 5.9 and 5.0 sit.

Here it is re-measured on all 14 ROIs × 5 seeds. If the band runs materially tighter elsewhere —
say 12–18 px — then `r5.0` sits above it too and the ladder is mis-centred, which is a reportable
finding rather than something to absorb quietly.

In [ ]:
G = LEDGER[LEDGER["axis"] == PRIMARY_AXIS]
print("same-object suppression gap (px) by dose -- distance from each mitotic annotation's")
print("nearest deep-pool local maximum to the peak that suppressed it:")
display(G.groupby("nms_radius_um")[["gap_n", "gap_p05_px", "gap_median_px", "gap_p95_px"]]
         .mean().round(2))
print("\nat the CONTROL radius, per domain -- this is the band the dose grid was positioned")
print("against, previously known only from 301.tiff seed 0 (reported there as 23-29 px):")
ctrl_gap = G[np.isclose(G["nms_radius_um"], 7.5)]
display(ctrl_gap.groupby("tumor_type")[["gap_n", "gap_p05_px", "gap_median_px", "gap_p95_px"]]
         .mean().round(2))
print("\nIf the p05-p95 band above sits BELOW the r5.0 radius (19.74-22.09 px), the ladder is")
print("mis-centred: every dose would then be suppressing the whole band and the doses could")
print("not differ for the registered reason.")

## 11. Summary

*(Numbers live in the cells above; this section records what they were asked to settle.)*

The three questions, in the order they were asked:

1. **Does shrinking help us surface the most TPs in the fewest candidates?** → §5 (the arbiter),
   §6 (the recall@K curves and K\*).
2. **Does the candidate list carry fewer FPs?** → §8 — and note that (1) and (2) are the *same
   question* at fixed K; only the full-list and reach-target framings are independent of it.
3. **Is any gain real, or geometry?** → §3 and §7 (coverage), §9 (the win/loss ledger), §10 (is
   the dose grid even centred on the mechanism it was built around).

**Limitations that survive this design**, stated so they are not rediscovered later:

* One template configuration (single 51 px, `TM_CCOEFF`, `hematoxylin_od`). The same-object gap is
  a property of the correlation surface, which is a property of the template — so the dose
  placement here does not transfer to a tightened template. That is F4's question, not this one.
* 14 ROIs is 2 per domain. The cluster tests treat the ROI as the unit and so are honest about it,
  but n = 14 bounds the resolution: the smallest attainable two-sided sign-flip p is 2/16384.
* recall@250 can be ceiling-pinned on the four ROIs carrying 17–19 evaluation mitoses (§4), which
  dilutes any pooled effect toward zero.